# RG3 부스팅 및 앙상블
기존26번 재사용, 신규 라이브러리 및 고정 교차모델 결합.

In [1]:
from pathlib import Path
import json
import xgboost,lightgbm,catboost,sktime,numba
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
root=Path.cwd()
if not (root/'data').exists():root=root.parent
src=''.join(json.loads((root/'notebooks/26_rg3_gas_15models.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
a=src.index('models={');b=src.index('trainrescache=')
models_text="""models={
'RandomForest':RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,class_weight='balanced_subsample',random_state=42,n_jobs=-1),
'ExtraTrees':ExtraTreesClassifier(n_estimators=200,max_depth=8,min_samples_leaf=3,class_weight='balanced',random_state=42,n_jobs=-1),
'XGBoost':XGBClassifier(n_estimators=200,max_depth=3,learning_rate=.03,min_child_weight=3,reg_lambda=5,subsample=.8,colsample_bytree=.8,eval_metric='logloss',random_state=42,n_jobs=4),
'LightGBM':LGBMClassifier(n_estimators=200,max_depth=3,num_leaves=7,min_child_samples=15,learning_rate=.03,reg_lambda=5,class_weight='balanced',random_state=42,n_jobs=4,verbosity=-1),
'CatBoost':CatBoostClassifier(iterations=200,depth=3,learning_rate=.03,l2_leaf_reg=5,auto_class_weights='Balanced',random_seed=42,thread_count=4,verbose=False,allow_writing_files=False)}
"""
src=src[:a]+models_text+src[b:]
src=src.split('# Verify exact old RF union')[0]
src=src.replace("'DecisionTree']:steps","'DecisionTree','XGBoost','LightGBM','CatBoost']:steps")
src=src.replace("    with warnings.catch_warnings(record=True) as ww:","    if name=='XGBoost':pipe.steps[-1][1].set_params(scale_pos_weight=float((yy==0).sum()/max((yy==1).sum(),1)))\n    with warnings.catch_warnings(record=True) as ww:")
src=src.replace('rg3_15models_','rg3_modern_')
src=src.replace("    steps=[SimpleImputer","    if name=='CatBoost' and pd.DataFrame(x).nunique(dropna=True).le(1).all():return np.full(len(q),.5)\n    steps=[SimpleImputer")
exec(src)
base=pp.copy()
ensemble_specs={'boost3_mean':(['XGBoost','LightGBM','CatBoost'],'mean'),'all5_mean':(list(models),'mean'),'all5_max':(list(models),'max')}
newrows=[]
for ens,(members,how) in ensemble_specs.items():
    for scheme in ['leave_one_block_out','past_blocks_only']:
        for j,block in enumerate(blocks):
            q=np.flatnonzero(groups.eq(block))
            ref=np.flatnonzero(~groups.eq(block)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
            if len(np.unique(y[ref]))<2:continue
            vals=np.column_stack([model_pair(name,ref,q).max(axis=1) for name in members])
            score=vals.mean(axis=1) if how=='mean' else vals.max(axis=1)
            inner=np.full((len(ref),len(members)),np.nan)
            for ib in groups.iloc[ref].unique():
                mask=groups.iloc[ref].eq(ib).to_numpy()
                if (~mask).any():
                    for k,name in enumerate(members):inner[mask,k]=model_pair(name,ref[~mask],ref[mask]).max(axis=1)
            cal=inner.mean(axis=1) if how=='mean' else inner.max(axis=1)
            normal=cal[(y[ref]==0)&np.isfinite(cal)]
            for budget in [.01,.03,.05]:
                cut=float(np.quantile(normal,1-budget,method='higher')) if len(normal) else 1.
                for i,z in enumerate(q):newrows.append(dict(model=ens,scheme=scheme,channel='cross_model_ensemble',budget=budget,TimeStamp=d.TimeStamp.iloc[z],block=block,actual=int(y[z]),score=float(score[i]),threshold=cut,prediction=int(score[i]>cut)))
pp=pd.concat([base,pd.DataFrame(newrows)],ignore_index=True);rows=[]
for (scheme,name,channel,budget),g in pp.groupby(['scheme','model','channel','budget'],sort=False):
    tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
    rows.append(dict(scheme=scheme,model=name,channel=channel,budget=budget,shots=len(g),gas=int(g.actual.sum()),AP=average_precision_score(g.actual,g.score),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),precision=tp/(tp+fp) if tp+fp else 0,recall=tp/(tp+fn)))
metrics=pd.DataFrame(rows)
metrics.to_csv(tables/'rg3_modern_metrics.csv',index=False,encoding='utf-8-sig')
pp.to_csv(tables/'rg3_modern_predictions.csv',index=False,encoding='utf-8-sig')
summary=metrics.loc[metrics.channel.isin(['max_union','cross_model_ensemble'])&metrics.budget.eq(.01)].sort_values(['scheme','AP'],ascending=[True,False])
versions={m.__name__:m.__version__ for m in [xgboost,lightgbm,catboost,sktime,numba]}
lines=['# RG3 최신 설치 모델과 교차 모델 앙상블','',str(versions),'',
'RH 정상566/가스17, 기존 두 신호(흔들림4·관계잔차4) 유지. 각 알고리즘의 두점수 max결합 이후 사전고정 부스팅3개 평균, 전체5개 평균 및 최대값을 비교한다. 비지도 방식이 아니다.',
'모델별 설정은 사전고정이며 XGBoost 불량가중치는 각 학습집합에서만 계산한다. 결합가중치는 동일하며 테스트 정답으로 선택하지 않는다. 모든 임계값은 학습집합 내부 구간검증 정상점수의1/3/5% 상위경계다. 모델별 점수는 완전 확률보정되지 않아 평균/max에는 척도영향이 있다.',
'평가구간은 센서 관측30분 단절 구간이며 정확한 실제 Run ID가 아니다. 전체구간 제외와 과거학습을 분리한다. 두 결과의 평가가스는17/13으로 다르다. 원본 이전센서이력 사용가능 조건이다. 동일자료 반복탐색이며 독립 최종평가가 아니다.',
'','| 평가 | 모델/결합 | AP | TP | FP | FN | TN |','|---|---|---:|---:|---:|---:|---:|']
for r in summary.itertuples():lines.append(f'| {r.scheme} | {r.model} | {r.AP:.4f} | {r.TP} | {r.FP} | {r.FN} | {r.TN} |')
(root/'reports/rg3_modern_ensemble_comparison.md').write_text('\n'.join(lines),encoding='utf-8')
print(summary.to_string(index=False));print(versions)


             scheme        model              channel  budget  shots  gas       AP  TP  FP  FN  TN  precision   recall
leave_one_block_out   ExtraTrees            max_union    0.01    583   17 0.174938   2  10  15 556   0.166667 0.117647
leave_one_block_out RandomForest            max_union    0.01    583   17 0.168625   4   6  13 560   0.400000 0.235294
leave_one_block_out     CatBoost            max_union    0.01    583   17 0.166562   2   5  15 561   0.285714 0.117647
leave_one_block_out    all5_mean cross_model_ensemble    0.01    583   17 0.139594   2   9  15 557   0.181818 0.117647
leave_one_block_out  boost3_mean cross_model_ensemble    0.01    583   17 0.091419   1   7  16 559   0.125000 0.058824
leave_one_block_out      XGBoost            max_union    0.01    583   17 0.088685   2   9  15 557   0.181818 0.117647
leave_one_block_out     all5_max cross_model_ensemble    0.01    583   17 0.077786   2  10  15 556   0.166667 0.117647
leave_one_block_out     LightGBM            max_